In [7]:
from pathlib import Path

import kagglehub
import pandas as pd

path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
orders = pd.read_csv(path / "olist_orders_dataset.csv", dtype="string")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

## Проверка и анализ olist_orders_dataset.csv

In [8]:
orders.shape

(99441, 8)

In [9]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [10]:
orders.info()
orders.head()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  string        
 1   customer_id                    99441 non-null  string        
 2   order_status                   99441 non-null  string        
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), string(3)
memory usage: 13.0 MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [50]:
orders[date_columns].agg(["min", "max"])

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12


In [11]:
orders['order_id'].nunique() == orders['order_id'].count()
#->нет дублирований по номеру заказа

np.True_

In [12]:
#проверка на отсутвие данных 
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [13]:
# проверка на наличие дублей
orders.duplicated().sum()

np.int64(0)

Проверка связи пропуска фактической доставки со статусом заказа

In [14]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64[pyarrow]

In [15]:
na_odcd = orders[orders['order_delivered_customer_date'].isna()]
na_odcd['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64[pyarrow]

Итого у нас 8 delivered нет даты получения и у 6 с canceled она есть 

Для расчёта доли опозданий берём заказы со статусом delivered, у которых есть корректные фактическая и обещанная даты.


In [16]:
is_canceled = orders["order_status"] == "canceled"
has_delivery_date = orders["order_delivered_customer_date"].notna()

orders[is_canceled & has_delivery_date]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


Проверка порядка событий. Получение раньше оформления 

In [17]:
invalid_order = (orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"])
invalid_order.sum()

0

Проверка порядка событий. Подтверждение раньше оформления 

In [18]:
invalid_order = (orders["order_status"] == "delivered") & (orders["order_approved_at"] < orders["order_purchase_timestamp"])
invalid_order.sum()

0

Проверка порядка событий. Передача перевозчику раньше подтверждения 

In [19]:
invalid_order = (orders["order_status"] == "delivered") & (orders["order_delivered_carrier_date"] < orders["order_approved_at"])
invalid_order.sum()

1350

Проверка порядка событий. Передача перевозчику раньше подтверждения  и Передача перевозчику раньше покупки

In [38]:
carrier_before_purchase = (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"])

both = invalid_order & carrier_before_purchase
print("2 нарушения дат", both.sum())
orders.loc[both, ["order_id", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date"]].head()

2 нарушения дат 165


,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date
615,b9afddbdcfadc9a87b41a83271c3e888,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00
1111,ad133696906f6a78826daa0911b7daec,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00
1329,74e033208dc13a7b8127eb8e73d09b76,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00
1372,a6b58794fd2ba533359a76c08df576e3,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00
1864,5792e0b1c8c8a2bf53af468c9a422c88,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00


Проверка порядка событий. Доставка раньше передачи

In [ ]:
delivery_before_carrier = (orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
delivery_before_carrier.sum()

23

In [ ]:
orders.loc[delivery_before_carrier, [
    "order_id",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]].head()

,order_id,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,2017-07-20 23:03:42,2017-07-20 18:52:41


Проверка порядка событий. Обещенный день доставки раньше дня оформления

In [ ]:
promes_delivery_before_purchase = orders['order_estimated_delivery_date'].dt.normalize() < orders['order_purchase_timestamp'].dt.normalize()
promes_delivery_before_purchase.sum()

np.int64(0)

## Итог olist_orders_dataset.csv

order_id уникален, полных дубликатов нет  
Диапазон покупок: с 2016-09-04 по 2018-10-17

### Есть n/a значения
order_approved_at                 160  
order_delivered_carrier_date     1783  
order_delivered_customer_date    2965  

### Проблемы со статусами и датами получения
У 8 заказов со статусом delivered нет даты получения  
У 6 заказов со статусом canceled дата получения заполнена  

### Порядок событий
1350 заказов — передача перевозчику раньше подтверждения  
в 165 из этих 1350 она раньше оформления  
23 заказа — получение раньше передачи перевозчику  
0 заказов — получение или обещанная дата раньше оформления

## Проверка и анализ olist_customers_dataset.csv

In [43]:
customers = pd.read_csv(path / "olist_customers_dataset.csv", dtype="string")
customers.info()
customers.head()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  string
 1   customer_unique_id        99441 non-null  string
 2   customer_zip_code_prefix  99441 non-null  string
 3   customer_city             99441 non-null  string
 4   customer_state            99441 non-null  string
dtypes: string(5)
memory usage: 11.5 MB


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [45]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [ ]:
customers['customer_id'].duplicated().sum()

np.int64(0)

In [ ]:
has_customers = orders['customer_id'].isin(customers['customer_id'])

has_customers.sum()


np.int64(99441)

In [ ]:
(~has_customers).sum() # отрицание 

np.int64(0)

In [ ]:
ids_per_customer = customers.groupby("customer_unique_id")["customer_id"].nunique()
ids_per_customer.head()
mul_ids = ids_per_customer > 1
mul_ids.sum()

np.int64(2997)

## Итог olist_customers_dataset.csv

customer_id уникален, полных дубликатов и n/a значений нет

### Связи и ключи
Все 99441 customer_id из orders есть в таблице customers  
У 2997 значений customer_unique_id есть несколько customer_id  
customer_id — ID  для конкретного заказа, customer_unique_id — ID самого покупателя 

## Проверка и анализ olist_order_items_dataset.csv

In [47]:
items = pd.read_csv(path / "olist_order_items_dataset.csv", dtype="string")
items.info()
items.head()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   order_id             112650 non-null  string
 1   order_item_id        112650 non-null  string
 2   product_id           112650 non-null  string
 3   seller_id            112650 non-null  string
 4   shipping_limit_date  112650 non-null  string
 5   price                112650 non-null  string
 6   freight_value        112650 non-null  string
dtypes: string(7)
memory usage: 19.6 MB


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
items['price'] = pd.to_numeric(items['price'])
items['freight_value'] = pd.to_numeric(items['freight_value'])
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

In [ ]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  string        
 1   order_item_id        112650 non-null  string        
 2   product_id           112650 non-null  string        
 3   seller_id            112650 non-null  string        
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  Float64       
 6   freight_value        112650 non-null  Float64       
dtypes: Float64(2), datetime64[us](1), string(4)
memory usage: 16.7 MB


In [ ]:
items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [49]:
items["order_id"].describe()

count                               112650
unique                               98666
top       8272b63d03f5f79c56e9e4120aec44ef
freq                                    21
Name: order_id, dtype: object

In [ ]:
items.duplicated().sum()

np.int64(0)

In [ ]:
items.duplicated(subset=["order_id", "order_item_id"]).sum()

np.int64(0)

In [ ]:
items['order_id'].isin(orders['order_id']).sum() # позиций с найденным заказом

np.int64(112650)

In [ ]:
orders['order_id'].isin(items['order_id']).sum() # заказы с позиций

np.int64(98666)

In [ ]:
(~orders["order_id"].isin(items["order_id"])).sum()  # заказы без позиций

np.int64(775)

In [53]:
products = pd.read_csv(path / "olist_products_dataset.csv", dtype="string")
sellers = pd.read_csv(path / "olist_sellers_dataset.csv", dtype="string")

In [ ]:
items['product_id'].isin(products['product_id']).sum()

np.int64(112650)

In [ ]:
items['seller_id'].isin(sellers['seller_id']).sum()

np.int64(112650)

In [ ]:
items["price"].describe()

count      112650.0
mean     120.653739
std      183.633928
min            0.85
25%            39.9
50%           74.99
75%           134.9
max          6735.0
Name: price, dtype: Float64

In [ ]:
items["freight_value"].describe()

count     112650.0
mean      19.99032
std      15.806405
min            0.0
25%          13.08
50%          16.26
75%          21.15
max         409.68
Name: freight_value, dtype: Float64

In [ ]:
(items["freight_value"] == 0).sum()

np.int64(383)

In [ ]:
items_in_order=items.groupby("order_id").size() # количество строк в группе
sellers_in_order = items.groupby("order_id")["seller_id"].nunique()

print("Заказов с несколькими позициями:", (items_in_order > 1).sum())
print("Заказов с одним продавцом:", (sellers_in_order == 1).sum())
print("Заказов с несколькими продавцами:", (sellers_in_order > 1).sum())


Заказов с несколькими позициями: 9803
Заказов с одним продавцом: 97388
Заказов с несколькими продавцами: 1278


In [ ]:
print("Стоимость всего:", round(items["price"].sum(), 2))
print("Стоимость всей доставки", round(items["freight_value"].sum(), 2))

Стоимость всего: 13591643.7
Стоимость всей доставки 2251909.54


In [66]:
# 1. статусы у 775 заказов без позиций
print(orders.loc[~orders["order_id"].isin(items["order_id"]), "order_status"].value_counts())

items_sellers = items.merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
print("Заказов с продавцами из разных штатов:", (items_sellers.groupby("order_id")["seller_state"].nunique() > 1).sum())

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64[pyarrow]
Заказов с продавцами из разных штатов: 510


## Итог olist_order_items_dataset.csv

order_id, order_item_id уникалены, полных дубликатов и n/a значений нет

### Связи с другими таблицами
Все product_id и seller_id найдены в справочниках товаров и продавцов  
У 775 заказов из orders нет позиций в items (статусы: unavailable - 603, canceled — 164, created — 5, invoiced — 2, shipped — 1; доставленных среди них нет)

### Структура заказов и суммы
Заказов с несколькими позициями: 9803  
Заказов с одним продавцом: 97388  
Заказов с несколькими продавцами: 1278 (из них у 510 продавцы из разных штатов)  
Стоимость всех товаров: 13591643.7  
Стоимость всей доставки: 2251909.54 (у 383 позиций доставка бесплатная = 0)

## Проверка и анализ olist_order_payments.csv

In [23]:
payments = pd.read_csv(path / "olist_order_payments_dataset.csv", dtype="string")
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   order_id              103886 non-null  string
 1   payment_sequential    103886 non-null  string
 2   payment_type          103886 non-null  string
 3   payment_installments  103886 non-null  string
 4   payment_value         103886 non-null  string
dtypes: string(5)
memory usage: 8.9 MB


In [ ]:
payments["payment_sequential"] = pd.to_numeric(payments["payment_sequential"])
payments["payment_installments"] = pd.to_numeric(payments["payment_installments"])
payments["payment_value"] = pd.to_numeric(payments["payment_value"])

In [ ]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [59]:
payments.duplicated(subset=["order_id", "payment_sequential"]).sum()

np.int64(0)

In [ ]:
payments['payment_value'].describe()

count      103886.0
mean      154.10038
std      217.494064
min             0.0
25%           56.79
50%           100.0
75%        171.8375
max        13664.08
Name: payment_value, dtype: Float64

In [24]:
payment_has_order = payments["order_id"].isin(orders["order_id"])

print("C", payment_has_order.sum())
print("Без", (~payment_has_order).sum())

C 103886
Без 0


In [25]:
order_has_payment = orders["order_id"].isin(payments["order_id"])

print("C", order_has_payment.sum())
print("Без", (~order_has_payment).sum())

C 99440
Без 1


In [ ]:
orders[~order_has_payment] #заказ, которого нет в payments

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [ ]:
payments_per_order = payments.groupby("order_id").size()
payment_types_per_order = (payments.groupby("order_id")["payment_type"].nunique())

print("Заказов с несколькими платёжными записями:",(payments_per_order > 1).sum())

print("Заказов с несколькими способами оплаты:",(payment_types_per_order > 1).sum())


Заказов с несколькими платёжными записями: 2961
Заказов с несколькими способами оплаты: 2246


In [ ]:
payments['payment_value'].sum()

np.float64(16008872.120000001)

In [ ]:
payments_per_order = payments.groupby("order_id")["payment_value"].sum()
payments_per_order

order_id
00010242fe8c5a6d1ba2dd792cb16214     72.19
00018f77f2f0320c557190d7a144bdd3    259.83
000229ec398224ef6ca0657da4fc703e    216.87
00024acbcdf0a6daa1e931b038114c75     25.78
00042b26cf59d7ce69dfabb4e55b4fd9    218.04
                                     ...  
fffc94f6ce00a00581880bf54a75a037     343.4
fffcd46ef2263f404302a634eb57f7eb    386.53
fffce4705a9662cd70adb13d4a31832d    116.85
fffe18544ffabc95dfada21779c9644f     64.71
fffe41c64501cc87c801fd61db3f6244     55.79
Name: payment_value, Length: 99440, dtype: Float64

In [ ]:
payments_per_order.sum()

np.float64(16008872.12)

In [41]:
payments.duplicated().sum()

np.int64(0)

## Итог olist_order_payments_dataset.csv

order_id, payment_sequential - уникалены, полных дубликатов и n/a значений нет

### Связи и структура оплат
У 1 доставленного заказа из orders (bfbd0f9bdef84302105ad712db648a6c) нет записи об оплате  
Заказов с несколькими платёжными записями: 2961  
Заказов с несколькими способами оплаты: 2246  
Общая сумма платежей: 16008872.12

## Проверка и анализ olist_order_reviews_dataset.csv

In [28]:
reviews = pd.read_csv(path / "olist_order_reviews_dataset.csv", dtype="string")

In [ ]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,<NA>,<NA>,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,<NA>,<NA>,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,<NA>,<NA>,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,<NA>,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,<NA>,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  string
 1   order_id                 99224 non-null  string
 2   review_score             99224 non-null  string
 3   review_comment_title     11568 non-null  string
 4   review_comment_message   40977 non-null  string
 5   review_creation_date     99224 non-null  string
 6   review_answer_timestamp  99224 non-null  string
dtypes: string(7)
memory usage: 17.9 MB


In [ ]:
reviews["review_score"] = pd.to_numeric(reviews["review_score"])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

In [ ]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [ ]:
reviews["review_id"].duplicated().sum()

np.int64(814)

In [ ]:
reviews["order_id"].duplicated().sum()

np.int64(551)

In [ ]:
reviews["review_score"].describe()

count     99224.0
mean     4.086421
std      1.347579
min           1.0
25%           4.0
50%           5.0
75%           5.0
max           5.0
Name: review_score, dtype: Float64

In [ ]:
(reviews["review_answer_timestamp"]< reviews["review_creation_date"]).sum()

np.int64(0)

In [ ]:
reviews["order_id"].count()

np.int64(99224)

In [30]:
review_has_order = reviews["order_id"].isin(orders["order_id"])

print("Отзывов с заказом", review_has_order.sum())
print("Отзывов без заказа", (~review_has_order).sum())

Отзывов с заказом 99224
Отзывов без заказа 0


In [61]:
rev_per_order = reviews.groupby("order_id")["review_score"].agg(["count", "nunique"])
print("Заказов с >1 отзывом:", (rev_per_order["count"] > 1).sum())
print("с разными оценками:", (rev_per_order["nunique"] > 1).sum())

Заказов с >1 отзывом: 547
с разными оценками: 202


## Итог olist_order_reviews_dataset.csv

Полных дубликатов нет, все order_id присутствуют в таблице orders

### Есть n/a значения
review_comment_title 87656  
review_comment_message 58247  

### Дубликаты ключей и логика
Найдено 814 неуникальных review_id и 551 повторный order_id  
Заказов с несколькими отзывами: 547 (из них у 202 оценки различаются)  
Оценок вне диапазона 1–5 нет  
Отзывов, где ответ записан раньше создания отзыва: 0

## Проверка и анализ olist_products_dataset.csv

In [ ]:
products = pd.read_csv(path / "olist_products_dataset.csv", dtype="string")

In [ ]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


In [ ]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   product_id                  32951 non-null  string
 1   product_category_name       32341 non-null  string
 2   product_name_lenght         32341 non-null  string
 3   product_description_lenght  32341 non-null  string
 4   product_photos_qty          32341 non-null  string
 5   product_weight_g            32949 non-null  string
 6   product_length_cm           32949 non-null  string
 7   product_height_cm           32949 non-null  string
 8   product_width_cm            32949 non-null  string
dtypes: string(9)
memory usage: 4.2 MB


In [ ]:
products["product_name_lenght"] = pd.to_numeric(products["product_name_lenght"])
products["product_description_lenght"] = pd.to_numeric(products["product_description_lenght"])
products["product_photos_qty"] = pd.to_numeric(products["product_photos_qty"])
products["product_weight_g"] = pd.to_numeric(products["product_weight_g"])
products["product_length_cm"] = pd.to_numeric(products["product_length_cm"])
products["product_height_cm"] = pd.to_numeric(products["product_height_cm"])
products["product_width_cm"] = pd.to_numeric(products["product_width_cm"])

In [ ]:
products.duplicated().sum()

np.int64(0)

In [ ]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.0,32341.0,32341.0,32949.0,32949.0,32949.0,32949.0
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.0,4.0,1.0,0.0,7.0,2.0,6.0
25%,42.0,339.0,1.0,300.0,18.0,8.0,15.0
50%,51.0,595.0,1.0,700.0,25.0,13.0,20.0
75%,57.0,972.0,3.0,1900.0,38.0,21.0,30.0
max,76.0,3992.0,20.0,40425.0,105.0,105.0,118.0


In [ ]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [ ]:
products["product_id"].describe()

count                                32951
unique                               32951
top       1e9e8ef04dbcff4541ed26657ea517e5
freq                                     1
Name: product_id, dtype: object

## Итог olist_products_dataset.csv

product_id уникален, полных дубликатов нет

### Есть n/a значения
product_category_name         610  
product_name_lenght           610  
product_description_lenght    610  
product_photos_qty            610  
product_weight_g                2  
product_length_cm               2  
product_height_cm               2  
product_width_cm                2

## Проверка и анализ olist_sellers_dataset.csv

In [ ]:
sellers = pd.read_csv(path / "olist_sellers_dataset.csv", dtype="string")

In [ ]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [ ]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   string
 1   seller_zip_code_prefix  3095 non-null   string
 2   seller_city             3095 non-null   string
 3   seller_state            3095 non-null   string
dtypes: string(4)
memory usage: 245.5 KB


In [ ]:
sellers.duplicated().sum()

np.int64(0)

In [ ]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [ ]:
sellers.describe()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
count,3095,3095,3095,3095
unique,3095,2246,611,23
top,3442f8959a84dea7ee197c632cb2df15,14940,sao paulo,SP
freq,1,49,694,1849


## Итог olist_sellers_dataset.csv

seller_id уникален, полных дубликатов и n/a значений нет

## Проверка и анализ olist_geolocation_dataset.csv

In [32]:
geolocation = pd.read_csv(path / "olist_geolocation_dataset.csv", dtype="string")

In [33]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1,01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
2,01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
3,01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
4,01035,-23.541577961711493,-46.64160722329613,sao paulo,SP


In [34]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype 
---  ------                       --------------    ----- 
 0   geolocation_zip_code_prefix  1000163 non-null  string
 1   geolocation_lat              1000163 non-null  string
 2   geolocation_lng              1000163 non-null  string
 3   geolocation_city             1000163 non-null  string
 4   geolocation_state            1000163 non-null  string
dtypes: string(5)
memory usage: 89.9 MB


In [35]:
geolocation["geolocation_lat"] = pd.to_numeric(geolocation["geolocation_lat"])
geolocation["geolocation_lng"] = pd.to_numeric(geolocation["geolocation_lng"])

In [ ]:
geolocation.duplicated().sum()

np.int64(261831)

In [ ]:
geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [54]:
print("Уникальные индексы", geolocation["geolocation_zip_code_prefix"].nunique())
print("Покупателей без индекса", (~customers["customer_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"])).sum())
print("Продавцов без индекса", (~sellers["seller_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"])).sum())

Уникальные индексы 19015
Покупателей без индекса 278
Продавцов без индекса 7


## Итог olist_geolocation_dataset.csv

n/a значений нет, найдено 261831 полных дубликатов

### Связи zip_code_prefix
Уникальных индексов: 19015  
Покупателей без индекса в geolocation: 278  
Продавцов без индекса в geolocation: 7

## Проверка и анализ olist_product_category_name_translation_dataset.csv

In [56]:
categories = pd.read_csv(path / "product_category_name_translation.csv",dtype="string")

In [ ]:
categories.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [ ]:
categories.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     string
 1   product_category_name_english  71 non-null     string
dtypes: string(2)
memory usage: 3.5 KB


In [ ]:
categories.duplicated().sum()

np.int64(0)

In [ ]:
categories.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [64]:
valid_cats = products["product_category_name"].dropna()
missing_cats = ~valid_cats.isin(categories["product_category_name"])
print("Товаров с категорией без перевода:", missing_cats.sum())
print("Категории без перевода:", valid_cats[missing_cats].unique())

Товаров с категорией без перевода: 13
Категории без перевода: <ArrowStringArray>
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 2, dtype: string


## Итог product_category_name_translation.csv

product_category_name уникален, дубликатов и n/a значений нет

### Связь с products
У 13 товаров из 2 категорий (pc_gamer, portateis_cozinha_e_preparadores_de_alimentos) отсутствует перевод в справочнике